# Text Chunking Strategies (Lesson 33)

Follow-along notebook for *Text chunking strategies* in Building with the Claude API.

No API calls in this one: chunking is plain Python. You'll run each strategy on the same sample report and compare the results, including the lesson's "bugs" problem, where a poor chunk boundary sends medical text to a software question.

**Run order:** Run All after a kernel restart.

## 1. Setup and a sample document

The lesson uses a company report with sections on several divisions. This sample (fictional) is built the same way, and deliberately uses the word *bug* in both the medical and software sections. It's also saved as `report.md` so later RAG lessons can reuse it.

In [1]:
import re
from collections import Counter
from math import sqrt
from pathlib import Path

report = """# Northwind Labs Annual Report 2026

## Executive Summary
Northwind Labs had a strong year across all divisions. Revenue grew 18 percent, driven by new contracts in diagnostics and cloud software. Headcount rose to 412 employees. This report summarizes results by division.

## Medical Research Division
The medical team completed Phase II trials for NL-204, a treatment for chronic inflammation. Enrollment was briefly paused in March after a stomach bug spread through one trial site. The bug was traced to a catering supplier and resolved within two weeks. Final results showed a 34 percent reduction in inflammation markers. Phase III planning begins next quarter.

## Software Engineering Division
Engineers shipped four major releases of the Northwind analytics platform this year. The team fixed 1,247 bugs, a 22 percent increase over last year, and cut average resolution time from nine days to four. A new automated test suite now covers 81 percent of the codebase. Customer-reported defects fell by half.

## Financial Overview
Total revenue reached 96 million dollars. Operating margin improved to 14 percent. Research spending was 21 million dollars, mostly in the medical division. Cash reserves at year end were 38 million dollars.

## Outlook
Next year the company plans to expand the analytics platform into healthcare, combining both divisions. Hiring will focus on data engineers and clinical researchers.
"""

Path("report.md").write_text(report, encoding="utf-8")
print(f"{len(report)} characters, {len(report.split())} words")

1424 characters, 220 words


A helper to print chunks with their lengths and boundaries, so the differences are easy to see:

In [2]:
def show_chunks(chunks, max_chars=None):
    print(f"{len(chunks)} chunks\n")
    for i, c in enumerate(chunks):
        text = c if max_chars is None else (c[:max_chars] + ("..." if len(c) > max_chars else ""))
        print(f"--- chunk {i} ({len(c)} chars) ---")
        print(repr(text))
        print()

## 2. Size-based chunking

Split into fixed-size pieces, with `chunk_overlap` characters shared between neighbors. Look at the start and end of each chunk: words and sentences get cut mid-way, and section headers can land at the end of one chunk while their content is in the next.

In [3]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        # step back by the overlap, unless we just reached the end
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)

    return chunks


show_chunks(chunk_by_char(report, chunk_size=150, chunk_overlap=20))

11 chunks

--- chunk 0 (150 chars) ---
'# Northwind Labs Annual Report 2026\n\n## Executive Summary\nNorthwind Labs had a strong year across all divisions. Revenue grew 18 percent, driven by ne'

--- chunk 1 (150 chars) ---
'ercent, driven by new contracts in diagnostics and cloud software. Headcount rose to 412 employees. This report summarizes results by division.\n\n## Me'

--- chunk 2 (150 chars) ---
' by division.\n\n## Medical Research Division\nThe medical team completed Phase II trials for NL-204, a treatment for chronic inflammation. Enrollment wa'

--- chunk 3 (150 chars) ---
'ation. Enrollment was briefly paused in March after a stomach bug spread through one trial site. The bug was traced to a catering supplier and resolve'

--- chunk 4 (150 chars) ---
'supplier and resolved within two weeks. Final results showed a 34 percent reduction in inflammation markers. Phase III planning begins next quarter.\n\n'

--- chunk 5 (150 chars) ---
'gins next quarter.\n\n## Software Engine

Try changing the overlap. With `0`, nothing is shared and cut-off words have no context; larger overlaps repeat more text (and cost more storage and tokens later).

In [4]:
for overlap in (0, 20, 50):
    chunks = chunk_by_char(report, chunk_size=150, chunk_overlap=overlap)
    print(f"overlap={overlap:>2}: {len(chunks)} chunks, {sum(len(c) for c in chunks)} total chars")

overlap= 0: 10 chunks, 1424 total chars
overlap=20: 11 chunks, 1624 total chars
overlap=50: 14 chunks, 2074 total chars


## 3. Structure-based chunking

Split on the document's own structure, here Markdown `## ` headers. Each chunk is one complete section.

The lesson's version splits *on* `\n## `, which removes the marker, so section titles lose their `## `. The second version uses a lookahead so the header stays attached to its content, which helps retrieval because the title often holds the best keywords.

In [5]:
def chunk_by_section(document_text):
    """Lesson version: the '## ' marker is consumed by the split."""
    return re.split(r"\n## ", document_text)


def chunk_by_section_keep_header(document_text):
    """Lookahead split: '## Title' stays at the top of each chunk."""
    return [c.strip() for c in re.split(r"\n(?=## )", document_text) if c.strip()]


print("Lesson version, first 60 chars of each chunk:")
for c in chunk_by_section(report):
    print("  ", repr(c[:60]))

print("\nHeader-preserving version:")
for c in chunk_by_section_keep_header(report):
    print("  ", repr(c[:60]))

Lesson version, first 60 chars of each chunk:
   '# Northwind Labs Annual Report 2026\n'
   'Executive Summary\nNorthwind Labs had a strong year across al'
   'Medical Research Division\nThe medical team completed Phase I'
   'Software Engineering Division\nEngineers shipped four major r'
   'Financial Overview\nTotal revenue reached 96 million dollars.'
   'Outlook\nNext year the company plans to expand the analytics '

Header-preserving version:
   '# Northwind Labs Annual Report 2026'
   '## Executive Summary\nNorthwind Labs had a strong year across'
   '## Medical Research Division\nThe medical team completed Phas'
   '## Software Engineering Division\nEngineers shipped four majo'
   '## Financial Overview\nTotal revenue reached 96 million dolla'
   '## Outlook\nNext year the company plans to expand the analyti'


## 4. Sentence-based chunking

Split into sentences with a regex, then group them with an overlap of whole sentences. A practical middle ground: no cut-off words, no need for structural guarantees.

Two small fixes to the lesson's version: it guards against `overlap_sentences >= max_sentences_per_chunk` (which would loop forever) and stops once the last sentence is included, so it doesn't emit a trailing chunk that's just a repeat of the overlap.

In [6]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    if overlap_sentences >= max_sentences_per_chunk:
        raise ValueError("overlap_sentences must be smaller than max_sentences_per_chunk")
    sentences = split_sentences(text)
    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        if end_idx == len(sentences):
            break
        start_idx += max_sentences_per_chunk - overlap_sentences

    return chunks


show_chunks(chunk_by_sentence(report, max_sentences_per_chunk=4, overlap_sentences=1), max_chars=160)

6 chunks

--- chunk 0 (273 chars) ---
'# Northwind Labs Annual Report 2026\n\n## Executive Summary\nNorthwind Labs had a strong year across all divisions. Revenue grew 18 percent, driven by new contract...'

--- chunk 1 (328 chars) ---
'This report summarizes results by division. ## Medical Research Division\nThe medical team completed Phase II trials for NL-204, a treatment for chronic inflamma...'

--- chunk 2 (299 chars) ---
'The bug was traced to a catering supplier and resolved within two weeks. Final results showed a 34 percent reduction in inflammation markers. Phase III planning...'

--- chunk 3 (344 chars) ---
'## Software Engineering Division\nEngineers shipped four major releases of the Northwind analytics platform this year. The team fixed 1,247 bugs, a 22 percent in...'

--- chunk 4 (218 chars) ---
'Customer-reported defects fell by half. ## Financial Overview\nTotal revenue reached 96 million dollars. Operating margin improved to 14 percent. Research spendi...'

--- chunk

Notice the regex only knows about `.`, `!`, `?`. Markdown headers have no punctuation, so `## Medical Research Division` gets glued onto the next sentence. Abbreviations like "Dr." or "e.g." would cause false splits. Real pipelines often use an NLP library's sentence splitter for this reason.

## 5. Semantic chunking (toy version)

The lesson describes semantic chunking without code because real implementations use embeddings, which is the next lesson. This toy version shows the *idea* with plain word overlap instead of embeddings:

1. Split into sentences
2. Score how similar each sentence is to the previous one
3. Start a new chunk where similarity drops below a threshold

It's crude (it only sees shared words, not meaning), but you can watch it find the topic shifts. Once you have embeddings, swap `similarity()` for cosine similarity of embedding vectors.

In [7]:
STOPWORDS = set("""a an and are as at be by for from has have in is it its of on or
that the this to was were will with after over into than then there these they""".split())


def word_vector(sentence):
    words = re.findall(r"[a-z]+", sentence.lower())
    return Counter(w for w in words if w not in STOPWORDS and len(w) > 2)


def similarity(a, b):
    va, vb = word_vector(a), word_vector(b)
    dot = sum(va[w] * vb[w] for w in va)
    norm = sqrt(sum(v * v for v in va.values())) * sqrt(sum(v * v for v in vb.values()))
    return dot / norm if norm else 0.0


def chunk_semantic_toy(text, threshold=0.1):
    sentences = split_sentences(text)
    chunks, current = [], [sentences[0]]
    for prev, sent in zip(sentences, sentences[1:]):
        score = similarity(prev, sent)
        if score < threshold:
            chunks.append(" ".join(current))
            current = [sent]
        else:
            current.append(sent)
    chunks.append(" ".join(current))
    return chunks


sentences = split_sentences(report)
for prev, sent in list(zip(sentences, sentences[1:]))[:8]:
    print(f"{similarity(prev, sent):.2f}  {sent[:70]!r}")

0.00  'Revenue grew 18 percent, driven by new contracts in diagnostics and cl'
0.00  'Headcount rose to 412 employees.'
0.00  'This report summarizes results by division.'
0.14  '## Medical Research Division\nThe medical team completed Phase II trial'
0.00  'Enrollment was briefly paused in March after a stomach bug spread thro'
0.11  'The bug was traced to a catering supplier and resolved within two week'
0.00  'Final results showed a 34 percent reduction in inflammation markers.'
0.00  'Phase III planning begins next quarter.'


In [8]:
show_chunks(chunk_semantic_toy(report, threshold=0.1), max_chars=120)

14 chunks

--- chunk 0 (112 chars) ---
'# Northwind Labs Annual Report 2026\n\n## Executive Summary\nNorthwind Labs had a strong year across all divisions.'

--- chunk 1 (83 chars) ---
'Revenue grew 18 percent, driven by new contracts in diagnostics and cloud software.'

--- chunk 2 (32 chars) ---
'Headcount rose to 412 employees.'

--- chunk 3 (165 chars) ---
'This report summarizes results by division. ## Medical Research Division\nThe medical team completed Phase II trials for ...'

--- chunk 4 (162 chars) ---
'Enrollment was briefly paused in March after a stomach bug spread through one trial site. The bug was traced to a cateri...'

--- chunk 5 (68 chars) ---
'Final results showed a 34 percent reduction in inflammation markers.'

--- chunk 6 (39 chars) ---
'Phase III planning begins next quarter.'

--- chunk 7 (238 chars) ---
'## Software Engineering Division\nEngineers shipped four major releases of the Northwind analytics platform this year. Th...'

--- chunk 8 (65 chars) ---
'A

## 6. The "bugs" problem

The lesson's warning: a user asks *"How many bugs did engineers fix this year?"* and gets medical text because that section also says "bug".

Here's a crude keyword retriever (the next lessons replace this with embeddings and BM25). For each strategy, it returns the best-matching chunk. Check whether the answer, **1,247**, is in it, and whether medical text came along.

In [ ]:
QUESTION = "How many bugs did engineers fix this year?"


def keyword_score(question, chunk):
    q = set(re.findall(r"[a-z]+", question.lower())) - STOPWORDS
    c = re.findall(r"[a-z]+", chunk.lower())
    # count matches, treating "bug" and "bugs" as the same word
    stem = lambda w: w[:-1] if w.endswith("s") else w
    q_stems = {stem(w) for w in q}
    return sum(1 for w in c if stem(w) in q_stems)


strategies = {
    "size (150/20)":       chunk_by_char(report, 150, 20),
    "size (400/50)":       chunk_by_char(report, 400, 50),
    "section":             chunk_by_section_keep_header(report),
    "sentence (4/1)":      chunk_by_sentence(report, 4, 1),
    "semantic (toy)":      chunk_semantic_toy(report, 0.1),
}

for name, chunks in strategies.items():
    best = max(chunks, key=lambda c: keyword_score(QUESTION, c))
    has_answer = "1,247" in best
    has_medical = "stomach" in best or "trial" in best
    print(f"{name:<16} answer={'yes' if has_answer else 'NO ':<3}  medical_text={'YES' if has_medical else 'no'}")
    print(f"                 {best[:110]!r}\n")

Experiment: change the chunk sizes, the overlap, or the threshold above and re-run this cell. Small size-based chunks are the most likely to return medical text or split "1,247 bugs" away from the engineering context.

## 7. Side-by-side summary

In [10]:
print(f"{'strategy':<16} {'chunks':>6} {'avg chars':>10} {'min':>5} {'max':>5}")
if "strategies" not in globals():
    strategies = {
        "size (150/20)": chunk_by_char(report, 150, 20),
        "size (400/50)": chunk_by_char(report, 400, 50),
        "section": chunk_by_section_keep_header(report),
        "sentence (4/1)": chunk_by_sentence(report, 4, 1),
        "semantic (toy)": chunk_semantic_toy(report, 0.1),
    }

print(f"{'strategy':<16} {'chunks':>6} {'avg chars':>10} {'min':>5} {'max':>5}")
for name, chunks in strategies.items():
    lengths = [len(c) for c in chunks]
    avg = sum(lengths) / len(lengths) if lengths else 0
    print(f"{name:<16} {len(chunks):>6} {avg:>10.0f} {min(lengths):>5} {max(lengths):>5}")

strategy         chunks  avg chars   min   max
strategy         chunks  avg chars   min   max
size (150/20)        11        148   124   150
size (400/50)         4        394   374   400
section               6        236    35   393
sentence (4/1)        6        294   218   344
semantic (toy)       14        100    32   239


## Takeaways
- **Structure-based** gives the cleanest chunks, but only when you control the document format.
- **Sentence-based** is a good middle ground for prose; its weak spot is sentence detection.
- **Size-based with overlap** is the reliable production default: works on anything, including code, and never fails outright.
- **Semantic** chunking follows meaning, at the cost of computing similarity (embeddings) for every sentence.
- Whatever you choose, test it against real questions, as in section 6, rather than judging chunks by eye.